# 최종 테이블 구성 (투자)
- DB: fintech_cohort_25_39
- 전체 고객: 13,852명
- 투자자: 4,079명
- 최신 데이터: 2026-10
- 최근 12개월: 2025-11 ~ 2026-10
- 투자 적극성: 12개월 투자이체액 / 가처분연소득 → 새 기간에서 33%, 67% 재계산
- 포트폴리오 HHI: Σ(종목별 매입금액 비중²) → ≤0.3 분산형 / 0.3~0.5 중간형 / >0.5 집중형
- 최종 결과: 기존과 동일한 10개 컬럼

In [1]:
from sqlalchemy import create_engine, inspect
from sqlalchemy.engine import URL
import pandas as pd
import numpy as np
import getpass

# 비밀번호 입력
password = getpass.getpass("MySQL 비밀번호: ")

# 새 DB
DB_NAME = "fintech_cohort_25_39"

# 연결 URL 생성
url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password=password,
    host="localhost",
    database=DB_NAME
)

# 엔진 생성
engine = create_engine(url)

# 연결 확인
inspector = inspect(engine)
tables = inspector.get_table_names()

print("DB:", DB_NAME)
print("전체 테이블 수:", len(tables))

print("\n테이블 목록:")
for table in tables:
    print("-", table)

MySQL 비밀번호:  ········


DB: fintech_cohort_25_39
전체 테이블 수: 25

테이블 목록:
- _cohort
- accounts
- card_billing_statements
- card_payments
- cards
- customer_credit_behavior
- customer_debt_summary
- customer_employment_events
- customer_financial_profiles
- customer_logins
- customer_monthly_cashflow
- customer_real_estate
- customer_stock_lots
- customer_stock_positions
- customers
- devices
- fraud_cases
- household_children
- investment_accounts
- loan_repayments
- loans
- mydata_consents
- real_assets
- self_employed_businesses
- transactions


In [2]:
# 연결 + 최신월 확인
# ============================================================
# 2. DB 및 분석기간 확인
# ============================================================

period_check = pd.read_sql("""
SELECT
    MIN(month) AS min_month,
    MAX(month) AS max_month,
    COUNT(DISTINCT month) AS month_count,
    COUNT(DISTINCT customer_id) AS customer_count
FROM customer_monthly_cashflow
""", engine)

print("=" * 60)
print("DATA PERIOD CHECK")
print("=" * 60)

display(period_check)

DATA PERIOD CHECK


,min_month,max_month,month_count,customer_count
0,2024-11,2026-10,24,13852


In [3]:
# ============================================================
# 3. 투자 적극성
# 최근 12개월: 2025-11 ~ 2026-10
# ============================================================

START_MONTH = "2025-11"
END_MONTH = "2026-10"

print("분석 기간:", START_MONTH, "~", END_MONTH)


# ------------------------------------------------------------
# 3-1. 투자자 기본 정보
# ------------------------------------------------------------

investor_base = pd.read_sql("""
SELECT
    customer_id,
    disposable_annual_income
FROM customer_financial_profiles
WHERE is_stock_investor = 1
""", engine)


# ------------------------------------------------------------
# 3-2. 최근 12개월 투자이체 합계
# ------------------------------------------------------------

investment_12m = pd.read_sql(f"""
SELECT
    customer_id,
    SUM(investment_transfer) AS investment_12m
FROM customer_monthly_cashflow
WHERE month BETWEEN '{START_MONTH}' AND '{END_MONTH}'
GROUP BY customer_id
""", engine)


# ------------------------------------------------------------
# 3-3. 투자자 기준 결합
# ------------------------------------------------------------

investment_feature = investor_base.merge(
    investment_12m,
    on="customer_id",
    how="left"
)

# 월 데이터가 없는 경우 0 처리
investment_feature["investment_12m"] = (
    investment_feature["investment_12m"].fillna(0)
)


# ------------------------------------------------------------
# 3-4. 투자 적극성 계산
# 최근 12개월 투자금액 / 가처분연소득
# ------------------------------------------------------------

investment_feature["investment_disposable_income_ratio"] = (
    investment_feature["investment_12m"]
    / investment_feature["disposable_annual_income"]
)


# ------------------------------------------------------------
# 3-5. Tertile 기준 계산
# ------------------------------------------------------------

ratio = investment_feature[
    "investment_disposable_income_ratio"
]

q33 = ratio.quantile(0.33)
q67 = ratio.quantile(0.67)


# ------------------------------------------------------------
# 3-6. 투자 적극성 유형 분류
# ------------------------------------------------------------

def classify_investment_intensity(x):
    if x <= q33:
        return "소극형"
    elif x <= q67:
        return "보통형"
    else:
        return "적극형"


investment_feature["investment_intensity_type"] = (
    ratio.apply(classify_investment_intensity)
)


# ============================================================
# 결과 확인
# ============================================================

print("=" * 65)
print("INVESTMENT INTENSITY - LATEST 12 MONTHS")
print("=" * 65)

print("분석 기간:", START_MONTH, "~", END_MONTH)
print("투자자 수:", len(investment_feature))
print("고객 ID 중복:", investment_feature["customer_id"].duplicated().sum())

print("\n[결측치]")
print(investment_feature.isna().sum())

print("\n[33%, 67% 기준]")
print("33%:", q33)
print("67%:", q67)

print("\n[투자 적극성 분포]")
display(
    ratio.describe(
        percentiles=[
            .01, .05, .10, .25, .33,
            .50, .67, .75, .90, .95, .99
        ]
    ).to_frame()
)

print("\n[유형별 인원]")
print(
    investment_feature[
        "investment_intensity_type"
    ].value_counts()
)

display(investment_feature.head())

분석 기간: 2025-11 ~ 2026-10
INVESTMENT INTENSITY - LATEST 12 MONTHS
분석 기간: 2025-11 ~ 2026-10
투자자 수: 4079
고객 ID 중복: 0

[결측치]
customer_id                           0
disposable_annual_income              0
investment_12m                        0
investment_disposable_income_ratio    0
investment_intensity_type             0
dtype: int64

[33%, 67% 기준]
33%: 0.046709999986945666
67%: 0.05930520015681111

[투자 적극성 분포]


,investment_disposable_income_ratio
count,4079.000000
mean,0.050266
std,0.017760
min,0.000000
1%,0.000000
5%,0.000000
10%,0.030132
25%,0.043425
33%,0.046710
50%,0.053430



[유형별 인원]
investment_intensity_type
보통형    1387
적극형    1346
소극형    1346
Name: count, dtype: int64


,customer_id,disposable_annual_income,investment_12m,investment_disposable_income_ratio,investment_intensity_type
0,18,51368860.25,3054906.12,0.059470,적극형
1,39,62830989.15,3554349.00,0.056570,보통형
2,44,25951381.22,1623898.60,0.062575,적극형
3,47,88321850.07,6918250.56,0.078330,적극형
4,161,61008168.57,3500648.76,0.057380,보통형


In [4]:
# ============================================================
# 4. 포트폴리오 집중도 (HHI)
# ============================================================

# ------------------------------------------------------------
# 4-1. 주식 보유 데이터
# ------------------------------------------------------------

stock_positions = pd.read_sql("""
SELECT
    customer_id,
    stock_code,
    quantity,
    purchase_price_actual_close
FROM customer_stock_positions
""", engine)


# ------------------------------------------------------------
# 4-2. 종목별 매입금액
# purchase_value = 수량 × 실제 매입가격
# ------------------------------------------------------------

stock_positions["purchase_value"] = (
    stock_positions["quantity"]
    * stock_positions["purchase_price_actual_close"]
)


# ------------------------------------------------------------
# 4-3. 고객-종목 단위 합산
# 동일 종목이 여러 행 존재하는 경우를 고려
# ------------------------------------------------------------

stock_by_customer = (
    stock_positions
    .groupby(
        ["customer_id", "stock_code"],
        as_index=False
    )
    .agg(
        purchase_value=("purchase_value", "sum")
    )
)


# ------------------------------------------------------------
# 4-4. 고객별 총 매입금액
# ------------------------------------------------------------

stock_by_customer["total_purchase_value"] = (
    stock_by_customer
    .groupby("customer_id")["purchase_value"]
    .transform("sum")
)


# ------------------------------------------------------------
# 4-5. 종목별 포트폴리오 비중
# ------------------------------------------------------------

stock_by_customer["weight"] = (
    stock_by_customer["purchase_value"]
    / stock_by_customer["total_purchase_value"]
)


# ------------------------------------------------------------
# 4-6. HHI 계산을 위한 비중 제곱
# ------------------------------------------------------------

stock_by_customer["weight_squared"] = (
    stock_by_customer["weight"] ** 2
)


# ------------------------------------------------------------
# 4-7. 고객별 HHI
# HHI = Σ(종목별 비중²)
# ------------------------------------------------------------

portfolio_feature = (
    stock_by_customer
    .groupby("customer_id", as_index=False)
    .agg(
        stock_count=("stock_code", "nunique"),
        total_purchase_value=("purchase_value", "sum"),
        portfolio_hhi=("weight_squared", "sum")
    )
)


# ------------------------------------------------------------
# 4-8. 논문 기준 집중도 분류
#
# HHI <= 0.3       : 분산형
# 0.3 < HHI <= 0.5 : 중간형
# HHI > 0.5        : 집중형
# ------------------------------------------------------------

def classify_portfolio_concentration(hhi):
    if hhi <= 0.3:
        return "분산형"
    elif hhi <= 0.5:
        return "중간형"
    else:
        return "집중형"


portfolio_feature["portfolio_concentration_type"] = (
    portfolio_feature["portfolio_hhi"]
    .apply(classify_portfolio_concentration)
)


# ============================================================
# 결과 확인
# ============================================================

print("=" * 65)
print("PORTFOLIO CONCENTRATION (HHI)")
print("=" * 65)

print("투자자 수:", len(portfolio_feature))
print(
    "고객 ID 중복:",
    portfolio_feature["customer_id"].duplicated().sum()
)

print("\n[결측치]")
print(portfolio_feature.isna().sum())


print("\n[HHI 분포]")
display(
    portfolio_feature["portfolio_hhi"]
    .describe(
        percentiles=[
            .01, .05, .10, .25,
            .50, .75, .90, .95, .99
        ]
    )
    .to_frame()
)


print("\n[집중도 유형별 결과]")

portfolio_summary = (
    portfolio_feature
    .groupby("portfolio_concentration_type")
    .agg(
        customer_count=("customer_id", "count"),
        mean_hhi=("portfolio_hhi", "mean"),
        median_hhi=("portfolio_hhi", "median"),
        mean_stock_count=("stock_count", "mean")
    )
)

portfolio_summary["percentage"] = (
    portfolio_summary["customer_count"]
    / len(portfolio_feature)
    * 100
)

portfolio_summary = portfolio_summary.reindex(
    ["분산형", "중간형", "집중형"]
)

display(portfolio_summary)


print("\n[HHI 범위 QA]")
print(
    "HHI <= 0:",
    (portfolio_feature["portfolio_hhi"] <= 0).sum()
)

print(
    "HHI > 1:",
    (portfolio_feature["portfolio_hhi"] > 1 + 1e-10).sum()
)

PORTFOLIO CONCENTRATION (HHI)
투자자 수: 4079
고객 ID 중복: 0

[결측치]
customer_id                     0
stock_count                     0
total_purchase_value            0
portfolio_hhi                   0
portfolio_concentration_type    0
dtype: int64

[HHI 분포]


,portfolio_hhi
count,4079.000000
mean,0.531245
std,0.246609
min,0.170869
1%,0.204821
5%,0.241840
10%,0.268927
25%,0.340968
50%,0.469967
75%,0.655894



[집중도 유형별 결과]


,customer_count,mean_hhi,median_hhi,mean_stock_count,percentage
portfolio_concentration_type,,,,,
분산형,662,0.255890,0.260628,5.132931,16.229468
중간형,1505,0.385712,0.376105,3.752159,36.896298
집중형,1912,0.741136,0.679716,1.980649,46.874234



[HHI 범위 QA]
HHI <= 0: 0
HHI > 1: 0


In [5]:
# ============================================================
# 5. FINAL INVESTMENT FEATURE TABLE
# ============================================================

# ------------------------------------------------------------
# 5-1. 전체 고객 기본 테이블
# ------------------------------------------------------------

final_feature = pd.read_sql("""
SELECT
    customer_id,
    is_stock_investor,
    disposable_annual_income
FROM customer_financial_profiles
""", engine)


# ------------------------------------------------------------
# 5-2. 투자 적극성 결합
# ------------------------------------------------------------

final_feature = final_feature.merge(
    investment_feature[
        [
            "customer_id",
            "investment_12m",
            "investment_disposable_income_ratio",
            "investment_intensity_type"
        ]
    ],
    on="customer_id",
    how="left"
)


# ------------------------------------------------------------
# 5-3. 포트폴리오 집중도 결합
# ------------------------------------------------------------

final_feature = final_feature.merge(
    portfolio_feature[
        [
            "customer_id",
            "stock_count",
            "total_purchase_value",
            "portfolio_hhi",
            "portfolio_concentration_type"
        ]
    ],
    on="customer_id",
    how="left"
)


# ------------------------------------------------------------
# 5-4. 비투자자 유형 처리
# 숫자형 변수 → NaN 유지
# 유형 변수 → 미해당
# ------------------------------------------------------------

final_feature["investment_intensity_type"] = (
    final_feature["investment_intensity_type"]
    .fillna("미해당")
)

final_feature["portfolio_concentration_type"] = (
    final_feature["portfolio_concentration_type"]
    .fillna("미해당")
)


# ------------------------------------------------------------
# 5-5. 최종 컬럼 순서
# ------------------------------------------------------------

final_feature = final_feature[
    [
        "customer_id",
        "is_stock_investor",
        "disposable_annual_income",
        "investment_12m",
        "investment_disposable_income_ratio",
        "investment_intensity_type",
        "stock_count",
        "total_purchase_value",
        "portfolio_hhi",
        "portfolio_concentration_type"
    ]
]


print("=" * 65)
print("FINAL INVESTMENT FEATURE TABLE")
print("=" * 65)

print("분석 기간:", START_MONTH, "~", END_MONTH)
print("테이블 크기:", final_feature.shape)

display(final_feature.head(10))

FINAL INVESTMENT FEATURE TABLE
분석 기간: 2025-11 ~ 2026-10
테이블 크기: (13852, 10)


,customer_id,is_stock_investor,disposable_annual_income,investment_12m,investment_disposable_income_ratio,investment_intensity_type,stock_count,total_purchase_value,portfolio_hhi,portfolio_concentration_type
0,18,1,51368860.25,3054906.12,0.059470,적극형,2.0,4212560.0,0.623672,집중형
1,39,1,62830989.15,3554349.00,0.056570,보통형,3.0,807175.0,0.546103,집중형
2,42,0,82366539.33,NaN,NaN,미해당,NaN,NaN,NaN,미해당
3,44,1,25951381.22,1623898.60,0.062575,적극형,2.0,6170360.0,0.836634,집중형
4,47,1,88321850.07,6918250.56,0.078330,적극형,3.0,1081276.0,0.476239,중간형
5,55,0,26614578.74,NaN,NaN,미해당,NaN,NaN,NaN,미해당
6,58,0,79881033.03,NaN,NaN,미해당,NaN,NaN,NaN,미해당
7,63,0,35532992.43,NaN,NaN,미해당,NaN,NaN,NaN,미해당
8,93,0,20674735.60,NaN,NaN,미해당,NaN,NaN,NaN,미해당
9,97,0,53789090.36,NaN,NaN,미해당,NaN,NaN,NaN,미해당


In [6]:
# ============================================================
# 6. FINAL QA
# ============================================================

print("=" * 65)
print("FINAL QA : INVESTMENT FEATURE TABLE")
print("=" * 65)


# 1. 기본 구조
print("\n[1. 기본 구조]")

print("분석 기간:", START_MONTH, "~", END_MONTH)
print("테이블 크기:", final_feature.shape)
print("전체 고객 수:", final_feature["customer_id"].nunique())
print("고객 ID 중복:", final_feature["customer_id"].duplicated().sum())

print(
    "투자자 수:",
    (final_feature["is_stock_investor"] == 1).sum()
)

print(
    "비투자자 수:",
    (final_feature["is_stock_investor"] == 0).sum()
)


# 2. 전체 결측치
print("\n[2. 전체 컬럼 결측치]")

print(final_feature.isna().sum())


# 3. 투자자 완전성
investor_final = final_feature[
    final_feature["is_stock_investor"] == 1
]

print("\n[3. 투자자 데이터 완전성]")

print(investor_final.isna().sum())


# 4. 비투자자 처리
non_investor_final = final_feature[
    final_feature["is_stock_investor"] == 0
]

print("\n[4. 비투자자 처리]")

print("비투자자 수:", len(non_investor_final))

print("\n숫자형 투자변수 non-null:")
print(
    non_investor_final[
        [
            "investment_12m",
            "investment_disposable_income_ratio",
            "stock_count",
            "total_purchase_value",
            "portfolio_hhi"
        ]
    ].notna().sum()
)

print("\n투자 적극성 유형:")
print(
    non_investor_final[
        "investment_intensity_type"
    ].value_counts()
)

print("\n포트폴리오 유형:")
print(
    non_investor_final[
        "portfolio_concentration_type"
    ].value_counts()
)


# 5. 최종 유형 분포
print("\n[5. 투자 적극성]")

print(
    investor_final[
        "investment_intensity_type"
    ].value_counts()
)

print("\n[6. 포트폴리오 집중도]")

print(
    investor_final[
        "portfolio_concentration_type"
    ].value_counts()
)


# 7. 값 범위
print("\n[7. 값 범위]")

print(
    "투자 적극성 < 0:",
    (
        investor_final[
            "investment_disposable_income_ratio"
        ] < 0
    ).sum()
)

print(
    "HHI <= 0:",
    (investor_final["portfolio_hhi"] <= 0).sum()
)

print(
    "HHI > 1:",
    (investor_final["portfolio_hhi"] > 1 + 1e-10).sum()
)

FINAL QA : INVESTMENT FEATURE TABLE

[1. 기본 구조]
분석 기간: 2025-11 ~ 2026-10
테이블 크기: (13852, 10)
전체 고객 수: 13852
고객 ID 중복: 0
투자자 수: 4079
비투자자 수: 9773

[2. 전체 컬럼 결측치]
customer_id                              0
is_stock_investor                        0
disposable_annual_income                 0
investment_12m                        9773
investment_disposable_income_ratio    9773
investment_intensity_type                0
stock_count                           9773
total_purchase_value                  9773
portfolio_hhi                         9773
portfolio_concentration_type             0
dtype: int64

[3. 투자자 데이터 완전성]
customer_id                           0
is_stock_investor                     0
disposable_annual_income              0
investment_12m                        0
investment_disposable_income_ratio    0
investment_intensity_type             0
stock_count                           0
total_purchase_value                  0
portfolio_hhi                         0
portfolio_concentr

# 최종 정리
| 항목 | 최종 기준 |
|---|---|
| DB | `fintech_cohort_25_39` |
| 전체 고객 | 13,852명 |
| 투자자 | 4,079명 |
| 비투자자 | 9,773명 |
| 투자 적극성 기간 | **2025-11 ~ 2026-10 (최근 12개월)** |
| 투자 적극성 | `investment_12m / disposable_annual_income` |
| 적극성 분류 | 33%, 67% tertile |
| 적극성 경계 | **0.046710 / 0.059305** |
| HHI | 종목별 매입금액 비중 제곱합 |
| HHI 분류 | ≤0.3 분산 / 0.3~0.5 중간 / >0.5 집중 |
| 최종 크기 | **13,852 × 10** |

In [7]:
# ============================================================
# 7. 최종 테이블 저장
# ============================================================

OUTPUT_FILE = "investment_feature_final_12m.csv"

final_feature.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", OUTPUT_FILE)
print("테이블 크기:", final_feature.shape)

저장 완료: investment_feature_final_12m.csv
테이블 크기: (13852, 10)
